# === DATASET ONE PREPROCESSING ===
This notebook houses the preprocessing steps for the first dataset. Preprocessing steps are as follows: 
1. **Perform dataset inspection** - Gathering information from shape, dtypes, nulls and duplicates. This helps in building a top level understanding of the dataset. 

2. **If there are nulls and duplicates present** - Handle them by removing the observation completely or re-write specific values via imputation. 

3. **Amend data types** - In order for the dataset to be compatabile with specific models, certain dataypes must be used. 

4. **Insepct and handle outliers or invalid values** - This stage involves determining whether or not there are any value present within the dataset that can be classified as unhelpful when deploying a model on the data (An example of this would be a negitive price or quantity amount).

5. **Standardise formats** - Organising aspects of the dataset like consistent casing and whitespace trimming. This ensures dataset robustness.

6. **Validate** - Performing a final check of shape, pulling a sample and confirming ratios will help to validate if the preprocessing steps have been successful. 

In [2]:
#Imports 
import pandas as pd
import numpy as np 
import matplotlib as plt 

# Import data and initilise a dataframe

In [3]:
#Import the dataset
dataset = pd.read_csv("../raw_data/Online_retail_One.csv", skiprows =1)

#Create a copy of the dataset in order to preseve to original. 
df = dataset.copy()

# Stage One
## Perform Top-Level dataset inspection: 
Here, we will begin to build an understanding of the overall look and behaviour of the dataset. 

In [4]:
# Print out the shape of the dataset: 
print("Dataset One Shape:", df.shape)

Dataset One Shape: (525461, 8)


Below we wil implement a function that produces a summary health report for the dataset: 

In [5]:
# Initlise the function: 
def health_summary(df):
    report = pd.DataFrame({
        'Data Type  ': df.dtypes,
        'Null Values  ': df.isnull().sum(),
        'Unique Values  ': df.nunique(),
        'Duplicate Values   ': df.duplicated().sum()
    })
    return report
print("-----------     Dataset One Health Summary     ------------")
display(health_summary(df))

-----------     Dataset One Health Summary     ------------


,Data Type,Null Values,Unique Values,Duplicate Values
Invoice,str,0,28816,6865
StockCode,str,0,4632,6865
Description,str,2928,4681,6865
Quantity,int64,0,825,6865
InvoiceDate,str,0,25296,6865
Price,float64,0,1606,6865
Customer ID,float64,107927,4383,6865
Country,str,0,40,6865


# Stage Two
## Handiling / Imputing Null values
The results from the health summary found in stage one highlights and issue with the data. There are two cateogies within the dataset that contain null values, they are 'Description' with 2928 null values and 'Customer ID' with 107,927 null values. Combined there are 110,855 null values within the dataset. 

This value displays that 21.1% of observations contain null values. In order to preserve the integrity of the dataset, these observation will be removed. This decision reflects and understanding of the data as removing null observations will still leave 414,606 observations for this dataset alone, still providing a comprehensive dataset. 

In [6]:
# Removing all null values from dataset: 
df = df.dropna()

# Stage Three
## Amending Data Types
In anticipation of the models that will be used within th emachine learning pipeline, we need to amend the datatypes found wihtin the dataset. Using K-means and DBSCAN means that all data needs to be numericals as both models are distance based, meaning strings and categories can not be used.

Also found within the code block below is a data type format inspection. This is a simple print function that displays a single observations of specific elements. This is done to aid correct data type conversion. For example 'Invoice Date'. For something like a date, choosing the correct strategy to split the date up into can be impactful on the final model. 

In [7]:
# Print an overview of the datatypes recorded for each category: 
print("-----------     Dataset One Data Types     ------------")
print("")
print(df.dtypes)

print("")
print("----   Dataset One Data Types Format Inspection   -----")
print("")
for column in df.columns:
    print(column,":", df[column].iloc[0])

-----------     Dataset One Data Types     ------------

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

----   Dataset One Data Types Format Inspection   -----

Invoice : 489434
StockCode : 85048
Description : 15CM CHRISTMAS GLASS BALL 20 LIGHTS
Quantity : 12
InvoiceDate : 12/1/09 7:45
Price : 6.95
Customer ID : 13085.0
Country : United Kingdom


As seen in the table above, there are five out of 8 categories that need to have thier data type changed. However, also seen in the results is that certain categories that are string data types actually contain integers. This makes conversion much simpler. The only categories that need a tailored approach are:

-  'Invoice Date' - Which will need intentionally pasring. 

- 'Country' - This can be dropped as it provides no relevance for consumer behavoir as well as not fitting in with the context of our hypothetical coffee shop as it is not international. This category can be dropped. 

- 'Description' - This category can also be dropped as it also provides no depth for the model to draw meaningful connections to concsumer behaviour. 

- 'Invoice' & 'StockCode' - As these features are strings, but a typical example of an observation shows only integers, it tells us that there may be a case of char's being used to denote particular behaviour (such as a cancellelation) within the data. This will need to be checked before changing these features to integer types. 

In [ ]:
# First, drop the two columns that are not neseccary for the ML models. 
#df = df.drop(['Country', 'Description'], axis = 1)

#Parse 'Invoive Date' into date time as first step
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format = '%m/%d/%y %H:%M')

# Implement day of week feature
df['day_of_week'] = df['InvoiceDate'].dt.dayofweek
# Implement hour feature
df['hour'] = df['InvoiceDate'].dt.hour
# Implement month feature
df['month'] = df['InvoiceDate'].dt.month

#Once encoding is complete drop 'Invoice Date' column
#df = df.drop(['Invoice Date'], axis = 1)

# Perform a check to determine if there are any non-conformative values within categories we are changing. 
invoice_outliers = df[df['Invoice'].astype(str).str.isnumeric() == False]['Invoice'].unique()
stockcode_outliers = df[df['StockCode'].astype(str).str.isnumeric() == False]['StockCode'].unique()

print("----     Dataset One Data Types Outliers Check     -----")
print("")
print("The sum of outlier values in Invoice is:", len(invoice_outliers))
print("The sum of outlier values in StockCode is:", len(stockcode_outliers))
print("The total sum of outlier values is:", len(stockcode_outliers) + len(invoice_outliers))
print("")
print("Example of an Invoice Outlier:", invoice_outliers[0])
print("Example of an StockCode Outlier:", stockcode_outliers[0])

# Before building the algorithm, check if there is outlier variance in both categories: 
# Find all the examples where a character is found within the value:
non_numeric_invoice = df[df['Invoice'].astype(str).str.contains(r'[^0-9]', regex = True)]['Invoice']
non_numeric_stock = df[df['StockCode'].astype(str).str.contains(r'[^0-9]', regex = True)]['StockCode']
#Within each of the results in non_numeric_chars, pull the specific chars found. 
invoice_unique_prefixes = non_numeric_invoice.str.extract(r'([^0-9]+)')[0].unique()
stock_unique_prefixes = non_numeric_stock.str.extract(r'([^0-9]+)')[0].unique()


# Print the reuslts: 
print("")
print("List of outlier characters in Invoice")
print(invoice_unique_prefixes)
print("")
print("List of outlier characters in StockCode:")
print(stock_unique_prefixes)

# Begin implementing the cancellations feature:
# Create a new column where each occurance of C wihtin Invoice is marked True
df['is_cancellation'] = df['Invoice'].astype(str).str.contains('C','c')

# Convert the two str features to integers: 
# 'Invoice' Conversion
df['Invoice'] = df['Invoice'].astype(str).str.replace('C','', regex = False).astype(int)
# 'StockCode' Conversion
df['StockCode'] = df['StockCode'].astype(str).str.replace('')
# Calculate cancellation rate per customer using newly formed cancellation data:
cancellation_rate = df.groupby('Customer ID')['is_cancellation'].mean()


----     Dataset One Data Types Outliers Check     -----

The sum of outlier values in Invoice is: 4372
The sum of outlier values in StockCode is: 1297
The total sum of outlier values is: 5669

Example of an Invoice Outlier: C489449
Example of an StockCode Outlier: 79323P

List of outlier characters in Invoice
<StringArray>
['C']
Length: 1, dtype: str

List of outlier characters in StockCode:
<StringArray>
[           'P',            'W',            'C',            'B',
            'F',            'L',            'S',            'A',
            'N',         'POST',            'E',            'J',
            'D',            'G',           'LP',           'BL',
            'K',            'H',           'GR',            'M',
            'U',            'R',            'V',            'T',
            'I', 'BANK CHARGES',            'O',            'Z',
         'TEST',         'PADS',       'ADJUST',           'SP',
            'Y']
Length: 33, dtype: str


There is an expansive list of results that are displayed above. First the unimportant features are dropped. Following this there is a check carried out on Invoice and StockCode used to identify any values that will interfer with the data type conversion (such as a character). Within this check it was confirmed that there were values that would have been uncooperative when converting the data type from string to integer. The sum of these ocurances were printed as well as examples of what the outliers looked like. This was followed by a check of what type of characters could be expected within the outlier. Results show that the only character to show up within the outliers was the letter 'C'. 

A decision was made to keep the outlier values. This decision reflects the importance of having cancellations within each customer profile as it adds another layer of behavioural segmentation to each ID. 

StockCode also produced some interesting results. There was a total of 33 different characters that were found across all the values. This produced a question for me to think about - Do I completely remove each instance of an character being present within each product code. For this, I chose not to remove the characters. Instead, I opted to preseve the feature columns as it is. I made this decision for two reasons. The first is that if I start removing characters from an extensive list of products I may introduce new more complex problems (IE 3589A and 3589B would both become 3589). Secondly, I relaised that I do no need to use this column specifically within the model pipeline. Instead, I could use this feature and its observations to derive another column called 'Unique Products Bought' This feature would be calculated per customer and already be and integer datatype.

In [16]:
# Create the new feature column - 'unique_products-bought'
unique_products_bought = df.groupby('Customer ID')['StockCode'].nunique()
# Example of new 
print(unique_products_bought.head())

Customer ID
12346.0    29
12347.0    70
12348.0    20
12349.0    92
12351.0    21
Name: StockCode, dtype: int64


In [9]:
# Confirm changes have been implemented correctly: 
print("")
print("----     Dataset One Data Types Removal Check     -----")
print("")
print(df.dtypes)


----     Dataset One Data Types Removal Check     -----

Invoice                     int64
StockCode                     str
Description                   str
Quantity                    int64
InvoiceDate        datetime64[us]
Price                     float64
Customer ID               float64
Country                       str
day_of_week                 int32
hour                        int32
month                       int32
is_cancellation              bool
dtype: object
